In [0]:
# ============================================================
# VB E-COMMERCE DATA ENGINEERING POC
# Gold Layer Transformations
#
# Topics:
# 1. SCD Type 1
# 2. SCD Type 2
# 3. MERGE / UPSERT
# 4. OPTIMIZE
# 5. ZORDER
# 6. Generic SCD2 Framework
# ============================================================

catalog_name = "vb_ecommerce"

print(f"Catalog: {catalog_name}")

Catalog: vb_ecommerce


In [0]:
%sql

USE CATALOG vb_ecommerce;

SHOW SCHEMAS;

databaseName
bronze
control
default
gold
information_schema
silver


In [0]:
%sql

SHOW TABLES IN gold;

database,tableName,isTemporary
gold,dim_customer,false
gold,dim_product,false
,_sqldf,true


In [0]:
%sql

SELECT *
FROM gold.dim_product
ORDER BY ProductKey;

ProductKey,ProductID,ProductName,Category,Price,StockQuantity,ProductStatus,CreatedDate,ModifiedDate
1,1,Laptop,Electronics,70000.00,50,Active,2026-08-28T22:23:46.173333Z,2026-08-30T11:06:50.405528Z
2,2,Mobile Phone,Electronics,30000.00,100,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
3,3,Headphones,Electronics,2500.00,200,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
4,4,Running Shoes,Sports,4500.00,75,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
5,5,Backpack,Accessories,1800.00,120,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z


## SCD1 Test

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW scd1_test_source AS

SELECT
    ProductID,
    ProductName,
    Category,
    CASE
        WHEN ProductID = 2 THEN 32000.00
        ELSE Price
    END AS Price,
    StockQuantity,
    CreatedDate,
    ModifiedDate,
    IsDeleted
FROM silver.products;

##SCD1 MERGE

In [0]:
%sql

MERGE INTO gold.dim_product AS target
USING silver.products AS source

ON target.ProductID = source.ProductID

WHEN MATCHED THEN
UPDATE SET
    target.ProductName   = source.ProductName,
    target.Category      = source.Category,
    target.Price         = source.Price,
    target.StockQuantity = source.StockQuantity,
    target.ModifiedDate  = source.ModifiedDate

WHEN NOT MATCHED THEN
INSERT
(
    ProductID,
    ProductName,
    Category,
    Price,
    StockQuantity,
    ProductStatus,
    CreatedDate,
    ModifiedDate
)
VALUES
(
    source.ProductID,
    source.ProductName,
    source.Category,
    source.Price,
    source.StockQuantity,
    CASE
        WHEN source.IsDeleted = true THEN 'Inactive'
        ELSE 'Active'
    END,
    source.CreatedDate,
    source.ModifiedDate
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


## VERIFY SCD1

In [0]:
%sql

SELECT
    ProductID,
    ProductName,
    Price
FROM scd1_test_source
ORDER BY ProductID;

ProductID,ProductName,Price
1,Laptop,70000.00
2,Mobile Phone,32000.00
3,Headphones,2500.00
4,Running Shoes,4500.00
5,Backpack,1800.00


In [0]:
%sql

SELECT
    ProductKey,
    ProductID,
    ProductName,
    Price,
    StockQuantity,
    ProductStatus,
    ModifiedDate
FROM gold.dim_product
ORDER BY ProductID;

ProductKey,ProductID,ProductName,Price,StockQuantity,ProductStatus,ModifiedDate
1,1,Laptop,70000.00,50,Active,2026-08-30T11:06:50.405528Z
2,2,Mobile Phone,30000.00,100,Active,2026-08-28T22:23:46.173333Z
3,3,Headphones,2500.00,200,Active,2026-08-28T22:23:46.173333Z
4,4,Running Shoes,4500.00,75,Active,2026-08-28T22:23:46.173333Z
5,5,Backpack,1800.00,120,Active,2026-08-28T22:23:46.173333Z


## SCD TYPE 2 

In [0]:
%sql

SELECT *
FROM gold.dim_customer
ORDER BY CustomerID, EffectiveStartDate;

CustomerKey,CustomerID,FirstName,LastName,Email,Phone,City,State,Country,CustomerStatus,EffectiveStartDate,EffectiveEndDate,IsCurrent,CreatedDate,ModifiedDate
1,1,Arun,Kumar,arun@example.com,9876543210,Trivandrum,Kerala,India,Active,2026-08-28,2026-08-30,false,2026-08-28T22:21:22.75Z,2026-08-30T11:14:33.613771Z
6,1,Arun,Kumar,arun@example.com,9876543210,Kochi,Kerala,India,Active,2026-08-30,9999-12-31,true,2026-08-30T11:16:47.10512Z,2026-08-30T11:16:47.10512Z
2,2,Priya,Nair,priya@example.com,9876543211,Kochi,Kerala,India,Active,2026-08-28,9999-12-31,true,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z
3,3,Rahul,Menon,rahul@example.com,9876543212,Chennai,Tamil Nadu,India,Active,2026-08-28,9999-12-31,true,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z
4,4,Sneha,Thomas,sneha@example.com,9876543213,Bengaluru,Karnataka,India,Active,2026-08-28,9999-12-31,true,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z
5,5,Vishnu,Raj,vishnu@example.com,9876543214,Coimbatore,Tamil Nadu,India,Active,2026-08-28,9999-12-31,true,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# GENERIC SCD TYPE 2
# Source  : silver.customers
# Target  : gold.dim_customer
# ============================================================

catalog = "vb_ecommerce"
source_table = f"{catalog}.silver.customers"
target_table = f"{catalog}.gold.dim_customer"

# 1. Read source
df_source = spark.table(source_table)

# 2. Ignore deleted records
df_source = df_source.filter(
    F.col("IsDeleted") == False
)

# 3. Current records from Gold
df_current = (
    spark.table(target_table)
    .filter(F.col("IsCurrent") == True)
)

# 4. Columns used to identify business changes
change_condition = (
    (F.coalesce(F.col("src.FirstName"), F.lit("")) !=
     F.coalesce(F.col("tgt.FirstName"), F.lit(""))) |
    (F.coalesce(F.col("src.LastName"), F.lit("")) !=
     F.coalesce(F.col("tgt.LastName"), F.lit(""))) |
    (F.coalesce(F.col("src.Email"), F.lit("")) !=
     F.coalesce(F.col("tgt.Email"), F.lit(""))) |
    (F.coalesce(F.col("src.Phone"), F.lit("")) !=
     F.coalesce(F.col("tgt.Phone"), F.lit(""))) |
    (F.coalesce(F.col("src.City"), F.lit("")) !=
     F.coalesce(F.col("tgt.City"), F.lit(""))) |
    (F.coalesce(F.col("src.State"), F.lit("")) !=
     F.coalesce(F.col("tgt.State"), F.lit(""))) |
    (F.coalesce(F.col("src.Country"), F.lit("")) !=
     F.coalesce(F.col("tgt.Country"), F.lit("")))
)

# 5. Identify new or changed customers
df_compare = (
    df_source.alias("src")
    .join(
        df_current.alias("tgt"),
        F.col("src.CustomerID") == F.col("tgt.CustomerID"),
        "left"
    )
)

df_changes = (
    df_compare
    .filter(
        F.col("tgt.CustomerID").isNull() |
        change_condition
    )
    .select("src.*")
)

print("New / Changed records:")
display(df_changes)

# 6. Close existing current versions
delta_target = DeltaTable.forName(
    spark,
    target_table
)

(
    delta_target.alias("tgt")
    .merge(
        df_changes.alias("src"),
        """
        tgt.CustomerID = src.CustomerID
        AND tgt.IsCurrent = true
        """
    )
    .whenMatchedUpdate(
        condition=change_condition,
        set={
            "EffectiveEndDate": "current_date()",
            "IsCurrent": "false",
            "ModifiedDate": "current_timestamp()"
        }
    )
    .execute()
)

# 7. Prepare new versions
df_new_versions = (
    df_changes
    .withColumn("EffectiveStartDate", F.current_date())
    .withColumn(
        "EffectiveEndDate",
        F.to_date(F.lit("9999-12-31"))
    )
    .withColumn("IsCurrent", F.lit(True))
)

# 8. Generate CustomerKey
max_key = (
    spark.table(target_table)
    .agg(F.max("CustomerKey"))
    .collect()[0][0]
)

max_key = max_key if max_key is not None else 0

df_new_versions = (
    df_new_versions
    .withColumn(
        "CustomerKey",
        F.monotonically_increasing_id() + max_key + 1
    )
)

# 9. Select target column order
df_new_versions = df_new_versions.select(
    "CustomerKey",
    "CustomerID",
    "FirstName",
    "LastName",
    "Email",
    "Phone",
    "City",
    "State",
    "Country",
    "CustomerStatus",
    "EffectiveStartDate",
    "EffectiveEndDate",
    "IsCurrent",
    "CreatedDate",
    "ModifiedDate"
)

# 10. Insert new versions
df_new_versions.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("SCD Type 2 processing completed successfully.")

New / Changed records:


CustomerID,FirstName,LastName,Email,Phone,City,State,Country,CreatedDate,ModifiedDate,IsDeleted


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8996167685787609>, line 145
    123 df_new_versions = df_new_versions.select(
    124     "CustomerKey",
    125     "CustomerID",
   (...)
    138     "ModifiedDate"
    139 )
    141 # 10. Insert new versions
    142 df_new_versions.write \
    143     .format("delta") \
    144     .mode("append") \
--> 145     .saveAsTable(target_table)
    147 print("SCD Type 2 processing completed successfully.")

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databr

In [0]:
%sql

SELECT
    CustomerKey,
    CustomerID,
    FirstName,
    City,
    CustomerStatus,
    EffectiveStartDate,
    EffectiveEndDate,
    IsCurrent,
    CreatedDate,
    ModifiedDate
FROM gold.dim_customer
WHERE CustomerID = 1
ORDER BY EffectiveStartDate;

CustomerKey,CustomerID,FirstName,City,CustomerStatus,EffectiveStartDate,EffectiveEndDate,IsCurrent,CreatedDate,ModifiedDate
1,1,Arun,Trivandrum,Active,2026-08-28,2026-08-30,false,2026-08-28T22:21:22.75Z,2026-08-30T11:14:33.613771Z
6,1,Arun,Kochi,Active,2026-08-30,9999-12-31,true,2026-08-30T11:16:47.10512Z,2026-08-30T11:16:47.10512Z


In [0]:
%sql

SELECT
    CustomerKey,
    CustomerID,
    FirstName,
    City,
    EffectiveStartDate,
    EffectiveEndDate,
    IsCurrent
FROM gold.dim_customer
ORDER BY CustomerID, EffectiveStartDate;

###SCD TYPE 2 Working

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# GENERIC SCD TYPE 2
# Source : silver.customers
# Target : gold.dim_customer
# ============================================================

catalog = "vb_ecommerce"

source_table = f"{catalog}.silver.customers"
target_table = f"{catalog}.gold.dim_customer"

print("Source :", source_table)
print("Target :", target_table)


# ============================================================
# 1. READ SOURCE
# ============================================================

df_source = spark.table(source_table)


# ============================================================
# 2. PREPARE SOURCE
#    CustomerStatus is derived from IsDeleted
# ============================================================

df_source = (
    df_source
    .withColumn(
        "CustomerStatus",
        F.when(F.col("IsDeleted") == True, "Inactive")
         .otherwise("Active")
    )
)


# ============================================================
# 3. READ CURRENT GOLD RECORDS
# ============================================================

df_current = (
    spark.table(target_table)
    .filter(F.col("IsCurrent") == True)
)


# ============================================================
# 4. CHANGE DETECTION
# ============================================================

change_condition = (
    (F.coalesce(F.col("src.FirstName"), F.lit("")) !=
     F.coalesce(F.col("tgt.FirstName"), F.lit("")))
    |
    (F.coalesce(F.col("src.LastName"), F.lit("")) !=
     F.coalesce(F.col("tgt.LastName"), F.lit("")))
    |
    (F.coalesce(F.col("src.Email"), F.lit("")) !=
     F.coalesce(F.col("tgt.Email"), F.lit("")))
    |
    (F.coalesce(F.col("src.Phone"), F.lit("")) !=
     F.coalesce(F.col("tgt.Phone"), F.lit("")))
    |
    (F.coalesce(F.col("src.City"), F.lit("")) !=
     F.coalesce(F.col("tgt.City"), F.lit("")))
    |
    (F.coalesce(F.col("src.State"), F.lit("")) !=
     F.coalesce(F.col("tgt.State"), F.lit("")))
    |
    (F.coalesce(F.col("src.Country"), F.lit("")) !=
     F.coalesce(F.col("tgt.Country"), F.lit("")))
    |
    (F.coalesce(F.col("src.CustomerStatus"), F.lit("")) !=
     F.coalesce(F.col("tgt.CustomerStatus"), F.lit("")))
)


# ============================================================
# 5. FIND NEW / CHANGED RECORDS
# ============================================================

df_compare = (
    df_source.alias("src")
    .join(
        df_current.alias("tgt"),
        F.col("src.CustomerID") == F.col("tgt.CustomerID"),
        "left"
    )
)

df_changes = (
    df_compare
    .filter(
        F.col("tgt.CustomerID").isNull()
        |
        change_condition
    )
    .select(
        "src.CustomerID",
        "src.FirstName",
        "src.LastName",
        "src.Email",
        "src.Phone",
        "src.City",
        "src.State",
        "src.Country",
        "src.CustomerStatus",
        "src.CreatedDate",
        "src.ModifiedDate"
    )
)

print("New / Changed records:")
display(df_changes)


# ============================================================
# 6. CLOSE EXISTING CURRENT RECORD
# ============================================================

delta_target = DeltaTable.forName(
    spark,
    target_table
)

(
    delta_target.alias("tgt")
    .merge(
        df_changes.alias("src"),
        """
        tgt.CustomerID = src.CustomerID
        AND tgt.IsCurrent = true
        """
    )
    .whenMatchedUpdate(
        condition=change_condition,
        set={
            "EffectiveEndDate": "current_date()",
            "IsCurrent": "false",
            "ModifiedDate": "current_timestamp()"
        }
    )
    .execute()
)


# ============================================================
# 7. CREATE NEW SCD2 VERSION
# ============================================================

df_new_versions = (
    df_changes
    .withColumn(
        "EffectiveStartDate",
        F.current_date()
    )
    .withColumn(
        "EffectiveEndDate",
        F.to_date(F.lit("9999-12-31"))
    )
    .withColumn(
        "IsCurrent",
        F.lit(True)
    )
)


# ============================================================
# 8. GENERATE SURROGATE CUSTOMER KEY
# ============================================================

max_key = (
    spark.table(target_table)
    .agg(F.max("CustomerKey"))
    .collect()[0][0]
)

if max_key is None:
    max_key = 0

df_new_versions = (
    df_new_versions
    .withColumn(
        "CustomerKey",
        F.row_number().over(
            Window.orderBy("CustomerID")
        ) + max_key
    )
)

# ============================================================
# 9. SELECT GOLD COLUMN ORDER
# ============================================================

df_new_versions = df_new_versions.select(
    "CustomerKey",
    "CustomerID",
    "FirstName",
    "LastName",
    "Email",
    "Phone",
    "City",
    "State",
    "Country",
    "CustomerStatus",
    "EffectiveStartDate",
    "EffectiveEndDate",
    "IsCurrent",
    "CreatedDate",
    "ModifiedDate"
)


# ============================================================
# 10. INSERT NEW VERSION
# ============================================================

df_new_versions = df_new_versions.select(
    "CustomerID",
    "FirstName",
    "LastName",
    "Email",
    "Phone",
    "City",
    "State",
    "Country",
    "CustomerStatus",
    "EffectiveStartDate",
    "EffectiveEndDate",
    "IsCurrent",
    "CreatedDate",
    "ModifiedDate"
)

df_new_versions.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("==========================================")
print("SCD TYPE 2 COMPLETED SUCCESSFULLY")
print("==========================================")

Source : vb_ecommerce.silver.customers
Target : vb_ecommerce.gold.dim_customer
New / Changed records:


CustomerID,FirstName,LastName,Email,Phone,City,State,Country,CustomerStatus,CreatedDate,ModifiedDate


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


SCD TYPE 2 COMPLETED SUCCESSFULLY


In [0]:
%sql

SELECT
    CustomerKey,
    CustomerID,
    FirstName,
    City,
    CustomerStatus,
    EffectiveStartDate,
    EffectiveEndDate,
    IsCurrent
FROM gold.dim_customer
ORDER BY CustomerID, EffectiveStartDate;

CustomerKey,CustomerID,FirstName,City,CustomerStatus,EffectiveStartDate,EffectiveEndDate,IsCurrent
1,1,Arun,Trivandrum,Active,2026-08-28,2026-08-30,false
6,1,Arun,Kochi,Active,2026-08-30,9999-12-31,true
2,2,Priya,Kochi,Active,2026-08-28,9999-12-31,true
3,3,Rahul,Chennai,Active,2026-08-28,9999-12-31,true
4,4,Sneha,Bengaluru,Active,2026-08-28,9999-12-31,true
5,5,Vishnu,Coimbatore,Active,2026-08-28,9999-12-31,true
